#   1. Importations  

In [1]:
import pandas as pd
import numpy as np
import json
import joblib


# Scikit-Learn - Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Scikit-Learn - Modèles
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# Scikit-Learn - Métriques
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuration
pd.set_option('display.max_columns', None)
SEED = 42

#   2. Chargement des données  

In [2]:
df = pd.read_csv('data/houses_Madrid_MASTER.csv')

# Nettoyage initial
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Dimensions du dataset : {df.shape}")
display(df.head())

Dimensions du dataset : (21742, 21)


,sq_mt_built,n_rooms,n_bathrooms,floor,buy_price,is_renewal_needed,is_new_development,built_year,has_ac,has_lift,is_exterior,has_garden,has_pool,has_terrace,has_storage_room,energy_certificate,has_parking,neighborhood,district,neigh_price_m2,house_type
0,64.0,2,1.0,3.0,85000,0,0,1960.0,1,0,1,0,0,0,0,4.0,0,San Cristóbal,Villaverde,1308.89,Pisos
1,70.0,3,1.0,4.0,129900,1,0,1970.0,0,1,1,0,0,1,0,3.0,0,Los Ángeles,Villaverde,1796.68,Pisos
2,94.0,2,2.0,1.0,144247,0,0,1970.0,0,1,1,0,0,0,1,3.0,0,San Andrés,Villaverde,1617.18,Pisos
3,64.0,2,1.0,0.0,109900,0,0,1955.0,0,1,1,0,0,0,1,3.0,0,San Andrés,Villaverde,1617.18,Pisos
4,108.0,2,2.0,4.0,260000,0,0,2003.0,1,1,1,0,1,0,1,3.0,1,Los Rosales,Villaverde,1827.79,Pisos


#   3. Preprocessing (Pipeline)  

In [3]:
# Définition des variables
target = 'buy_price'
X = df.drop(columns=[target])
y = df[target]

# Identification des types de colonnes
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

print(f"Variables numériques ({len(numeric_features)}) : {list(numeric_features)}")
print(f"Variables catégorielles ({len(categorical_features)}) : {list(categorical_features)}")

# Pipeline Numérique : Imputation (médiane) + Standardisation
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline Catégoriel : Imputation (constante) + OneHotEncoding
# handle_unknown='ignore' est crucial pour la production (si un nouveau quartier apparaît)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Le processeur global
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("\nPipeline de preprocessing créé.")

Variables numériques (17) : ['sq_mt_built', 'n_rooms', 'n_bathrooms', 'floor', 'is_renewal_needed', 'is_new_development', 'built_year', 'has_ac', 'has_lift', 'is_exterior', 'has_garden', 'has_pool', 'has_terrace', 'has_storage_room', 'energy_certificate', 'has_parking', 'neigh_price_m2']
Variables catégorielles (3) : ['neighborhood', 'district', 'house_type']

Pipeline de preprocessing créé.


#   4. Séparation Train / Test  

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print(f"Entraînement sur {X_train.shape[0]} lignes.")
print(f"Test sur {X_test.shape[0]} lignes.")

Entraînement sur 17393 lignes.
Test sur 4349 lignes.


# 5. Recherche des meilleurs modèles (GridSearch)

In [5]:
best_estimators = {}
model_scores = []

#   A. Linear Regression  
print("\n  1/3 Optimisation Linear Regression  ")
pipe_lr = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
param_lr = {'regressor__fit_intercept': [True, False]}

grid_lr = GridSearchCV(pipe_lr, param_lr, cv=5, scoring='r2', n_jobs=-1)
grid_lr.fit(X_train, y_train)

best_estimators['LinearRegression'] = grid_lr.best_estimator_
print(f"Best R2: {grid_lr.best_score_:.4f}")


#   B. Random Forest  
print("\n  2/3 Optimisation Random Forest  ")
pipe_rf = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=SEED))])
param_rf = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipe_rf, param_rf, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)

best_estimators['RandomForest'] = grid_rf.best_estimator_
print(f"Best R2: {grid_rf.best_score_:.4f}")


#   C. SVR (Support Vector Regressor)  

print("\n  3/3 Optimisation SVR ")
pipe_svr = Pipeline([('preprocessor', preprocessor), ('regressor', SVR())])
param_svr = {
    'regressor__C': [1, 10],      
    'regressor__kernel': ['rbf']   
}


grid_svr = GridSearchCV(pipe_svr, param_svr, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_svr.fit(X_train[:5000], y_train[:5000])

best_estimators['SVR'] = grid_svr.best_estimator_
print(f"Best R2 (sur subset): {grid_svr.best_score_:.4f}")


  1/3 Optimisation Linear Regression  
Best R2: 0.7656

  2/3 Optimisation Random Forest  
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best R2: 0.9062

  3/3 Optimisation SVR (Patience...)  
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best R2 (sur subset): -0.1260


# 6. Évaluation et Choix du "Champion"

In [6]:
results_data = []

for name, model in best_estimators.items():
    # Prédiction
    y_pred = model.predict(X_test)
    
    # Métriques
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results_data.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "Params": model.named_steps['regressor'].get_params()
    })

results_df = pd.DataFrame(results_data).sort_values(by='R2', ascending=False)
display(results_df)


best_model_name = results_df.iloc[0]['Model']
best_pipeline = best_estimators[best_model_name]
best_metrics = results_df.iloc[0].to_dict()

print(f"\n LE MEILLEUR MODÈLE EST : {best_model_name}")

,Model,R2,RMSE,MAE,Params
1,RandomForest,0.915243,218940.008794,95158.654457,"{'bootstrap': True, 'ccp_alpha': 0.0, 'criteri..."
0,LinearRegression,0.786319,347632.046529,194818.975127,"{'copy_X': True, 'fit_intercept': False, 'n_jo..."
2,SVR,-0.144005,804360.533277,447280.085911,"{'C': 10, 'cache_size': 200, 'coef0': 0.0, 'de..."



 LE MEILLEUR MODÈLE EST : RandomForest


# 7. Entraînement Final & Sauvegarde

In [7]:
name_mapping = {
    "LinearRegression": "Linear Regression",
    "RandomForest": "Random Forest",
    "SVR": "SVR"
}

final_json_structure = {}

print("Début de la finalisation des modèles...\n")

for original_name, model in best_estimators.items():
    name = name_mapping.get(original_name, original_name)
    model_key = name.lower().replace(" ", "_")
    
    artifact_name = f"{model_key}_model.joblib"
    
    print(f"Traitement de : {name} (Fichier cible : {artifact_name})")
    
    #   A. Calcul des métriques sur le TEST SET
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    #   B. Extraction des paramètres  
    regressor_params = model.named_steps['regressor'].get_params()
    filtered_params = {}
    
    if "Linear" in original_name:
        filtered_params["fit_intercept"] = regressor_params.get("fit_intercept")
        filtered_params["normalize"] = regressor_params.get("normalize", False) 
        
    elif "Forest" in original_name:
        filtered_params["n_estimators"] = regressor_params.get("n_estimators")
        filtered_params["max_depth"] = regressor_params.get("max_depth")
        filtered_params["min_samples_split"] = regressor_params.get("min_samples_split")
        filtered_params["min_samples_leaf"] = regressor_params.get("min_samples_leaf")
        
    elif "SVR" in original_name:
        filtered_params["C"] = regressor_params.get("C")
        filtered_params["epsilon"] = regressor_params.get("epsilon")
        filtered_params["kernel"] = regressor_params.get("kernel")

    #   C. Ré-entraînement sur 100% des données  
    print("Ré-entraînement sur tout le dataset...")
    model.fit(X, y)
    
    #   D. Sauvegarde Joblib  
    joblib.dump(model, artifact_name)
    print(f"Modèle sauvegardé : {artifact_name}\n\n")
    
    #   E. Ajout au JSON  
    final_json_structure[model_key] = {
        "name": name,
        "artifact_name": artifact_name,
        "metrics": {
            "rmse": round(rmse, 2),
            "mae": round(mae, 2),
            "r2": round(r2, 4)
        },
        "parameters": filtered_params
    }

print("Fichier JSON\n\n")
print(json.dumps(final_json_structure, indent=2))

Début de la finalisation des modèles...

Traitement de : Linear Regression (Fichier cible : linear_regression_model.joblib)
Ré-entraînement sur tout le dataset...
Modèle sauvegardé : linear_regression_model.joblib


Traitement de : Random Forest (Fichier cible : random_forest_model.joblib)
Ré-entraînement sur tout le dataset...
Modèle sauvegardé : random_forest_model.joblib


Traitement de : SVR (Fichier cible : svr_model.joblib)
Ré-entraînement sur tout le dataset...
Modèle sauvegardé : svr_model.joblib


Fichier JSON


{
  "linear_regression": {
    "name": "Linear Regression",
    "artifact_name": "linear_regression_model.joblib",
    "metrics": {
      "rmse": 347632.05,
      "mae": 194818.98,
      "r2": 0.7863
    },
    "parameters": {
      "fit_intercept": false,
      "normalize": false
    }
  },
  "random_forest": {
    "name": "Random Forest",
    "artifact_name": "random_forest_model.joblib",
    "metrics": {
      "rmse": 218940.01,
      "mae": 95158.65,
      "r2": 0.